# Customer Churn & Retention Analytics — Step 1: Inspect the Dataset

First step before touching anything else — just getting a feel for what's actually in this dataset. Checking shape, data types, missing values, duplicates, and how balanced the target variable is.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.shape

(7043, 21)

7,043 rows and 21 columns. Let's look at the first few rows to get a sense of what we're working with.

In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Data types and nulls

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


No nulls according to `.info()`, but that's suspicious for `TotalCharges` — it's showing up as an object (string) column instead of numeric, which usually means there's some non-numeric junk hiding in there (blank strings, etc). Checking that directly.

In [4]:
# TotalCharges is being read as a string, so pd.to_numeric will flag anything that can't convert
bad_total_charges = df[pd.to_numeric(df['TotalCharges'], errors='coerce').isna()]
print(f"Rows where TotalCharges can't convert to a number: {len(bad_total_charges)}")
bad_total_charges[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]

Rows where TotalCharges can't convert to a number: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


All 11 of them have `tenure = 0` — these are brand new customers who haven't been billed yet, so `TotalCharges` is just blank rather than actually missing data. Good to know for the cleaning step next.

## Duplicate rows / duplicate customer IDs

In [5]:
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate customerIDs:", df['customerID'].duplicated().sum())

Fully duplicate rows: 0
Duplicate customerIDs: 0


No duplicates. Each row is a unique customer.

## Target variable balance — Churn

In [6]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
pd.DataFrame({'count': churn_counts, 'pct': churn_pct.round(2)})

,count,pct
Churn,,
No,5174,73.46
Yes,1869,26.54


About 73.5% No / 26.5% Yes. Not a crazy imbalance, but enough that I'll want to keep an eye on it later when building the ML model (accuracy alone won't be a useful metric — recall/precision on the churn class will matter more).

## Unique values per categorical column

Just want to make sure there's no weird inconsistent labeling (e.g. 'Yes'/'yes'/'Y') before I start grouping by these columns later.

In [7]:
# excluding customerID (just an identifier) and the numeric columns
exclude = ['customerID', 'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical_cols = [c for c in df.columns if c not in exclude]

for col in categorical_cols:
    print(f"{col}: {df[col].unique()}")

gender: ['Female' 'Male']
Partner: ['Yes' 'No']
Dependents: ['No' 'Yes']
PhoneService: ['No' 'Yes']
MultipleLines: ['No phone service' 'No' 'Yes']
InternetService: ['DSL' 'Fiber optic' 'No']
OnlineSecurity: ['No' 'Yes' 'No internet service']
OnlineBackup: ['Yes' 'No' 'No internet service']
DeviceProtection: ['No' 'Yes' 'No internet service']
TechSupport: ['No' 'Yes' 'No internet service']
StreamingTV: ['No' 'Yes' 'No internet service']
StreamingMovies: ['No' 'Yes' 'No internet service']
Contract: ['Month-to-month' 'One year' 'Two year']
PaperlessBilling: ['Yes' 'No']
PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
Churn: ['No' 'Yes']


Labeling is clean and consistent. A few columns (`MultipleLines`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) have a third value like `'No internet service'` or `'No phone service'` — that's not messy data, it's just a real category (customers without internet/phone can't have those add-ons). Worth remembering for the cleaning/EDA steps — I may want to simplify these to Yes/No or keep the third category depending on what the analysis needs.

## Numeric column summary

In [8]:
numeric_cols = ['tenure', 'MonthlyCharges']
df[numeric_cols].describe()

,tenure,MonthlyCharges
count,7043.000000,7043.000000
mean,32.371149,64.761692
std,24.559481,30.090047
min,0.000000,18.250000
25%,9.000000,35.500000
50%,29.000000,70.350000
75%,55.000000,89.850000
max,72.000000,118.750000


- `tenure` ranges 0-72 months (6 years), median around 29 months
- `MonthlyCharges` ranges about $18-$119, pretty wide spread

`TotalCharges` isn't included here since it's still stored as a string — that gets fixed in the cleaning step.

## Summary of what I found

- 7,043 customers, 21 columns, no duplicate rows
- `TotalCharges` needs to be converted from string to numeric — 11 rows have a blank value, all with `tenure = 0` (brand new customers), so these should become 0, not get dropped
- No inconsistent categorical labeling
- Several service columns have a legitimate third category (`No internet service` / `No phone service`) rather than just Yes/No
- Target variable (`Churn`) is moderately imbalanced: 73.5% No / 26.5% Yes — something to account for when building the ML model later
- `SeniorCitizen` is stored as 0/1 instead of Yes/No like the other Yes/No columns — worth standardizing in cleaning

Next step: clean the dataset based on what I found here.